# LendMind

### 1) Importing Libraries

In [ ]:
!pip install numpy==2.2.0
!pip install pandas==2.2.3
!pip install matplotlib==3.9.3

In [23]:
import numpy as np 
import pandas as pd
from matplotlib import pyplot as plt

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

### 2) Read & Understand The Data

In [24]:
all_data = pd.read_csv(r"..\DataSet\archive\loan.csv")

In [25]:
all_data.shape

(2260668, 145)

A) We will take a random sample from the dataset (all_data), for example 5% of the total dataset:

no of rows = 0.05 * 2260668 = 113,033 rows

--> a)

In [26]:
all_data = all_data[all_data['loan_status'].isin(['Fully Paid', 'Charged Off'])]
all_data['loan_status'] = all_data['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})
all_data.shape

(1303607, 145)

--> b)

In [27]:
df = all_data.sample(frac= 0.05, random_state= 42)
df.shape

(65180, 145)

In [37]:
df.sample(5)

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,debt_settlement_flag
885075,4200,36 months,11.44,B,B4,10+ years,MORTGAGE,65000.0,Source Verified,0,...,2.0,77.1,0.0,0.0,0.0,65167.0,6500.0,63630.0,Cash,N
1090899,3000,36 months,17.57,D,D4,10+ years,RENT,40000.0,Source Verified,0,...,9.0,100.0,42.9,0.0,0.0,9564.0,13100.0,2007.0,Cash,N
1011534,5600,36 months,7.89,A,A5,10+ years,MORTGAGE,62000.0,Verified,0,...,2.0,91.7,0.0,0.0,0.0,25318.0,21600.0,17255.0,Cash,N
624366,25000,36 months,17.27,D,D2,8 years,MORTGAGE,72000.0,Verified,1,...,2.0,100.0,100.0,0.0,0.0,65830.0,7000.0,63430.0,Cash,N
1358191,10000,36 months,10.49,B,B2,< 1 year,OWN,80000.0,Not Verified,0,...,2.0,100.0,0.0,0.0,0.0,36484.0,30000.0,37177.0,Cash,N


B) dealing with Null values

In [29]:
df = df.loc[:, df.isnull().mean()<0.4]
df.shape

(65180, 87)

In [36]:
df.sample(5)

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,debt_settlement_flag
1602906,10000,36 months,9.93,B,B2,9 years,RENT,80600.00,Not Verified,0,...,0.0,100.0,28.6,0.0,0.0,16141.0,19300.0,24474.0,Cash,N
700876,24000,60 months,16.29,D,D1,10+ years,MORTGAGE,125000.00,Not Verified,0,...,1.0,100.0,100.0,0.0,0.0,135159.0,46600.0,80531.0,Cash,N
1694441,5000,36 months,19.20,D,D3,NaN,MORTGAGE,31599.84,Verified,0,...,2.0,95.2,50.0,1.0,0.0,28874.0,9400.0,24722.0,Cash,N
962576,12000,60 months,18.99,E,E3,10+ years,MORTGAGE,45000.00,Verified,1,...,2.0,96.8,100.0,0.0,0.0,83783.0,61500.0,12870.0,Cash,Y
1800160,12000,36 months,17.77,D,D1,5 years,RENT,90000.00,Not Verified,0,...,2.0,96.0,50.0,0.0,0.0,75183.0,22600.0,61883.0,Cash,N


C) Remove columns where all cells have the same value (Zero Variance Filter)

In [31]:
df = df.loc[:, df.nunique()>1]
df.shape

(65180, 82)

In [35]:
df.sample(5)

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,debt_settlement_flag
777962,21000,60 months,10.75,B,B4,10+ years,RENT,42000.0,Source Verified,1,...,2.0,100.0,10.0,0.0,0.0,47151.0,52800.0,35720.0,Cash,N
2224969,11000,36 months,7.21,A,A3,2 years,RENT,78800.0,Source Verified,0,...,0.0,100.0,0.0,0.0,0.0,2849.0,5000.0,16500.0,Cash,N
1443180,5000,36 months,11.49,B,B5,7 years,MORTGAGE,60000.0,Source Verified,1,...,4.0,100.0,50.0,0.0,0.0,47048.0,12600.0,47426.0,Cash,Y
1207093,10000,36 months,7.89,A,A5,2 years,RENT,52000.0,Source Verified,1,...,1.0,100.0,0.0,1.0,1.0,6703.0,11800.0,7000.0,Cash,N
1997018,17100,36 months,13.98,C,C3,6 years,OWN,47650.0,Verified,0,...,5.0,100.0,33.3,0.0,0.0,47716.0,32000.0,34896.0,Cash,N


D) High Cardinality Filter

In [33]:
string_columns = df.select_dtypes(include='object').columns
string_df = df[string_columns]
hated_columns = string_df.loc[:, (string_df.nunique() > 50)].columns
df = df.drop(columns=hated_columns)
df.shape

(65180, 75)

E) Multicollinearity Filter

In [34]:
corr_matrix = df.select_dtypes(include=['number']).corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]
df = df.drop(columns=to_drop)
df.shape

(65180, 66)

F) Preventing Data Leakage